# Combining Causal Designs

**Module 06 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement ITS+DiD (controlled interrupted time series)
2. Implement fuzzy RDD as an IV design
3. Use a design selection decision tree to choose the appropriate estimator
4. Understand when combining designs strengthens identification

## Why Combine Designs?

Each causal design relies on specific assumptions:
- ITS: no confounding events at treatment time
- DiD: parallel trends
- RDD: continuity at cutoff
- IV: exclusion restriction

Combining designs can triangulate an estimate and make identification more credible when individual designs have limitations.

In [ ]:
learning_objectives(["Implement ITS+DiD (controlled interrupted time series)", "Implement fuzzy RDD as an IV design", "Use a design selection decision tree to choose the appropriate estimator", "Understand when combining designs strengthens identification"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Controlled ITS (ITS + DiD)

**Setting:** A state introduces a new tobacco excise tax in year 2015. We have data for 20 years (2005-2024) across 50 states. 15 states adopt the tax in 2015; 35 states never adopt.

**ITS weakness:** Without a comparison group, we can't rule out confounding national trends (e.g., general decline in smoking rates)

**ITS+DiD solution:** Use non-adopting states as a control group to remove common time trends

In [ ]:
# Generate tobacco policy panel data
N_STATES = 50
N_TREATED = 15
TREAT_YEAR = 2015
YEARS = list(range(2005, 2025))
TRUE_EFFECT = -3.5  # percentage point decline in smoking rate

rows = []
for s in range(N_STATES):
    treated = 1 if s < N_TREATED else 0
    state_fe = np.random.normal(0, 4)   # baseline differences between states
    state_trend = np.random.normal(-0.3, 0.1)  # state-specific declining trend
    
    for year in YEARS:
        t = year - 2005  # time trend (0-19)
        post = 1 if year >= TREAT_YEAR else 0
        
        # Common national trend: smoking declining everywhere
        national_trend = -0.5 * t
        # State-specific time trend
        state_specific = state_trend * t
        # Treatment effect (only post-adoption for treated states)
        tau = TRUE_EFFECT * post * treated
        
        smoking_rate = (25 + state_fe + national_trend + state_specific
                        + tau + np.random.normal(0, 1.5))
        rows.append({
            'state': s, 'year': year, 'treated': treated, 'post': post,
            'post_treated': post * treated, 'smoking_rate': smoking_rate,
            'time': t
        })

df_panel = pd.DataFrame(rows)

print(f"Panel: {N_STATES} states × {len(YEARS)} years = {len(df_panel)} obs")
print(f"Treated states: {N_TREATED} | Control states: {N_STATES - N_TREATED}")
print(f"True treatment effect: {TRUE_EFFECT} pp decline in smoking rate")

In [ ]:
# Method 1: Pure ITS (treated states only, no control)
treated_df = df_panel[df_panel['treated'] == 1].copy()
its_model = smf.ols(
    'smoking_rate ~ time + post + post:time',
    data=treated_df
).fit(cov_type='cluster', cov_kwds={'groups': treated_df['state']})
its_effect = its_model.params['post']

# Method 2: Simple DiD (no time trends)
did_model = smf.ols(
    'smoking_rate ~ C(state) + C(year) + post_treated',
    data=df_panel
).fit(cov_type='cluster', cov_kwds={'groups': df_panel['state']})
did_effect = did_model.params['post_treated']

# Method 3: Controlled ITS (ITS + DiD)
# Include state FE, year FE, AND state-specific time trends
its_did_model = smf.ols(
    'smoking_rate ~ C(state) + C(year) + post_treated + time:C(treated)',
    data=df_panel
).fit(cov_type='cluster', cov_kwds={'groups': df_panel['state']})
its_did_effect = its_did_model.params['post_treated']

print("=" * 60)
print("TOBACCO TAX POLICY: ESTIMATION COMPARISON")
print("=" * 60)
print(f"{'Method':<35} {'Estimate':>10}")
print("-" * 60)
print(f"{'True effect':<35} {TRUE_EFFECT:>+10.3f}")
print(f"{'Pure ITS (no control group)':<35} {its_effect:>+10.3f}")
print(f"{'DiD (TWFE)':<35} {did_effect:>+10.3f}")
print(f"{'Controlled ITS (ITS+DiD)':<35} {its_did_effect:>+10.3f}")
print("-" * 60)
print()
print("Controlled ITS combines:")
print("  - Control group (removes common national decline trend)")
print("  - State-specific time trends (removes differential pre-trends)")
print("  - Post-treatment indicator (isolates policy effect)")

In [ ]:
# Visualise: treated vs control trends
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: group means over time
ax = axes[0]
for group, label, color in [(1, 'Treated (tax adopters)', 'steelblue'),
                              (0, 'Control (never adopted)', 'darkorange')]:
    means = df_panel[df_panel['treated']==group].groupby('year')['smoking_rate'].mean()
    ax.plot(means.index, means.values, 'o-', color=color, linewidth=2, markersize=4, label=label)

ax.axvline(TREAT_YEAR - 0.5, color='red', linestyle='--', linewidth=2, label='Tax adoption (treated)')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Smoking Rate (%)')
ax.set_title('Controlled ITS: Smoking Rates Over Time')
ax.legend()

# Right: method comparison
ax2 = axes[1]
methods = ['True Effect', 'Pure ITS', 'DiD', 'Controlled ITS']
estimates = [TRUE_EFFECT, its_effect, did_effect, its_did_effect]
colors_bar = ['green', 'gray', 'steelblue', 'darkorange']

bars = ax2.barh(range(len(methods)), estimates, color=colors_bar, alpha=0.7, height=0.5)
ax2.axvline(TRUE_EFFECT, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_yticks(range(len(methods)))
ax2.set_yticklabels(methods)
ax2.set_xlabel('Treatment Effect Estimate (pp)')
ax2.set_title('Method Comparison: Which Gets Closest?')
for bar, est in zip(bars, estimates):
    ax2.text(est - 0.1, bar.get_y() + bar.get_height()/2, f'{est:.2f}',
             ha='right', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 2. Fuzzy RDD as IV

**Setting:** A job training program is targeted at workers with risk scores above 70 out of 100. However, compliance is imperfect: some workers above 70 don't enroll (60% compliance), and some below 70 self-enroll through other channels (5% always-taker rate).

The cutoff serves as an instrument: we estimate the effect using the local jump in enrollment probability at the cutoff.

In [ ]:
N = 1500
CUTOFF = 70
COMPLIANCE = 0.60    # probability eligible workers enroll
ALWAYS_TAKER = 0.05  # probability ineligible workers self-enroll
TRUE_EFFECT_FUZZY = 15.0  # wage increase for compliers

# Risk score (running variable)
score = np.random.uniform(40, 100, N)
x = score - CUTOFF  # centred
eligible = (score >= CUTOFF).astype(int)

# Actual enrollment (fuzzy compliance)
enrolled = np.where(
    eligible == 1,
    np.random.binomial(1, COMPLIANCE, N),      # eligible: 60% enroll
    np.random.binomial(1, ALWAYS_TAKER, N)     # ineligible: 5% enroll
)

# Wage: depends on enrollment (causal) and score (endogenous via risk level)
wage = (20 + 0.2 * score + TRUE_EFFECT_FUZZY * enrolled
        + np.random.normal(0, 8, N))

df_fuzzy = pd.DataFrame({'score': score, 'x': x, 'eligible': eligible,
                          'enrolled': enrolled, 'wage': wage})

# First stage: jump in enrollment at cutoff
bw = 15
local = df_fuzzy[np.abs(df_fuzzy['x']) <= bw].copy()

fs_model = smf.ols('enrolled ~ eligible + x + eligible:x', data=local).fit(cov_type='HC1')
jump_fs = fs_model.params['eligible']

# Reduced form: jump in wage at cutoff
rf_model = smf.ols('wage ~ eligible + x + eligible:x', data=local).fit(cov_type='HC1')
jump_rf = rf_model.params['eligible']

# Fuzzy RDD (Wald)
tau_fuzzy = jump_rf / jump_fs

print("Fuzzy RDD Analysis:")
print("=" * 50)
print(f"First stage jump (enrollment at cutoff): {jump_fs:.3f}")
print(f"  Enrollment above cutoff: {local[local['eligible']==1]['enrolled'].mean():.3f}")
print(f"  Enrollment below cutoff: {local[local['eligible']==0]['enrolled'].mean():.3f}")
print()
print(f"Reduced form jump (wage at cutoff): {jump_rf:.3f}")
print()
print(f"Fuzzy RDD (Wald) estimate: {jump_rf:.3f} / {jump_fs:.3f} = {tau_fuzzy:.2f}")
print(f"True LATE for compliers:   {TRUE_EFFECT_FUZZY:.2f}")

In [ ]:
# Visualise fuzzy RDD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: enrollment rate (first stage)
ax = axes[0]
n_bins = 15
local_all = df_fuzzy[np.abs(df_fuzzy['x']) <= bw + 10]

for side, color in [(local_all[local_all['x'] < 0], 'steelblue'),
                     (local_all[local_all['x'] >= 0], 'darkorange')]:
    side_copy = side.copy()
    side_copy['bin'] = pd.cut(side_copy['x'], bins=8)
    bin_means = side_copy.groupby('bin')['enrolled'].mean()
    x_mids = [iv.mid for iv in bin_means.index]
    ax.scatter(x_mids, bin_means.values, color=color, s=50, zorder=5)
    if len(x_mids) > 1:
        z = np.polyfit(x_mids, bin_means.values, 1)
        ax.plot(np.linspace(min(x_mids), max(x_mids), 100),
                np.polyval(z, np.linspace(min(x_mids), max(x_mids), 100)),
                '-', color=color, linewidth=2)

ax.axvline(0, color='red', ls='--', linewidth=2)
ax.set_xlabel('Score (centred at cutoff = 70)')
ax.set_ylabel('Enrollment Rate')
ax.set_title(f'First Stage: Jump in Enrollment = {jump_fs:.2f}\n(60% compliance → jump < 1)')

# Right: wage outcome (reduced form)
ax2 = axes[1]
for side, color in [(local_all[local_all['x'] < 0], 'steelblue'),
                     (local_all[local_all['x'] >= 0], 'darkorange')]:
    side_copy = side.copy()
    side_copy['bin'] = pd.cut(side_copy['x'], bins=8)
    bin_means = side_copy.groupby('bin')['wage'].mean()
    x_mids = [iv.mid for iv in bin_means.index]
    ax2.scatter(x_mids, bin_means.values, color=color, s=50, zorder=5)
    if len(x_mids) > 1:
        z = np.polyfit(x_mids, bin_means.values, 1)
        ax2.plot(np.linspace(min(x_mids), max(x_mids), 100),
                 np.polyval(z, np.linspace(min(x_mids), max(x_mids), 100)),
                 '-', color=color, linewidth=2)

ax2.axvline(0, color='red', ls='--', linewidth=2)
ax2.set_xlabel('Score (centred at cutoff = 70)')
ax2.set_ylabel('Wage')
ax2.set_title(f'Reduced Form: Jump in Wage = {jump_rf:.2f}\n(Fuzzy LATE = {tau_fuzzy:.2f})')

plt.suptitle('Fuzzy RDD = IV Design\nFirst Stage × LATE = Reduced Form', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Design Selection Decision Tree

In [ ]:
print("CAUSAL DESIGN SELECTION GUIDE")
print("=" * 60)

# Simulate a set of research scenarios and determine appropriate design
scenarios = [
    {
        "question": "Effect of a new drug (randomly assigned in RCT)",
        "randomised": True,
        "threshold": False,
        "comparison_group": True,
        "panel_data": True,
        "recommended_design": "RCT (gold standard)",
        "note": "Randomisation gives clean identification"
    },
    {
        "question": "Effect of scholarship on graduation (score cutoff)",
        "randomised": False,
        "threshold": True,
        "comparison_group": False,
        "panel_data": False,
        "recommended_design": "Sharp RDD",
        "note": "Exact cutoff → sharp RDD; check density and covariate balance"
    },
    {
        "question": "Effect of minimum wage on employment (state-level policy)",
        "randomised": False,
        "threshold": False,
        "comparison_group": True,
        "panel_data": True,
        "recommended_design": "DiD or Controlled ITS",
        "note": "Panel + comparison group → DiD; check parallel trends"
    },
    {
        "question": "Return to college education (observational)",
        "randomised": False,
        "threshold": False,
        "comparison_group": False,
        "panel_data": False,
        "recommended_design": "IV (need instrument)",
        "note": "No natural experiment → need exogenous variation; college proximity is classic"
    },
    {
        "question": "Effect of air quality regulation on health (single city, time series)",
        "randomised": False,
        "threshold": False,
        "comparison_group": False,
        "panel_data": True,
        "recommended_design": "ITS",
        "note": "No comparison group → ITS; identify confounders carefully"
    },
]

for s in scenarios:
    print(f"\nQuestion: {s['question']}")
    print(f"  Recommended design: {s['recommended_design']}")
    print(f"  Note: {s['note']}")

print("\n" + "=" * 60)
print("KEY PRINCIPLE: Use the most credible design available.")
print("Each design is only as strong as its identifying assumption.")

## Self-Check

1. In the Controlled ITS example, add a global shock in 2015 that increases smoking rates by 1 percentage point for all states (an economic shock during treatment year). Does Pure ITS detect this as a treatment effect? Does Controlled ITS handle it correctly?

2. In the Fuzzy RDD, increase `ALWAYS_TAKER` to 0.30 (30% of ineligible workers self-enroll). How does this change the first stage jump? How does it change the fuzzy LATE estimate?

3. For the tobacco panel: run the event study for treated vs control states to verify parallel pre-trends. Do you find evidence of parallel trends before 2015?

4. Apply the design selection guide to one of your own research questions or a policy question you care about. Which design is most appropriate and why?

---
**Previous:** `02_weak_instruments.ipynb`
**Next:** Module 07 — Production Pipelines

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])